# Variational Auto Encoder(VAE)의 구현

## 0. 환경 설정

Python >= 3.7, pytorch >= 1.3.1

In [ ]:
import os, numpy as np
import torch
from torch import nn, optim
from tqdm.notebook import tqdm
from torchvision.utils import save_image, make_grid

In [ ]:
dataset_path = "./datasets"
batch_size = 64
x_dim = 784
hidden_dim = 400
latent_dim = 200
lr = 1e-3
epochs = 30
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
mnist_transform = transforms.Compose([transforms.ToTensor()])
train_dataset = MNIST(dataset_path, transform=mnist_transform, train=True,
                      download=True)
test_dataset = MNIST(dataset_path, transform=mnist_transform, train=False,
                      download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size)

In [ ]:
# Gaussian MLP Encoder and Decoder기반의 단순한 구현
from torch import nn, optim
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.FC_input = nn.Linear(input_dim, hidden_dim)
        self.FC_input2 = nn.Linear(hidden_dim, hidden_dim)
        self.FC_mean = nn.Linear(hidden_dim, latent_dim)
        self.FC_var = nn.Linear(hidden_dim, latent_dim)
        self.LeakyReLU = nn.LeakyReLU(.2) # 음수 기울기 .01-->.2
        self.training = True
    def forward(self, x):
        h_ = self.LeakyReLU(self.FC_input(x))
        h_ = self.LeakyReLU(self.FC_input2(h_))
        mean = self.FC_mean(h_) # Encoder에서 평균 및 분산을 생성
        log_var = self.FC_var(h_) # "q" 정규 분포 에 따르는 단순한 구
        return mean, log_var

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.FC_hidden = nn.Linear(latent_dim, hidden_dim)
        self.FC_hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.FC_output = nn.Linear(hidden_dim, output_dim)
        self.LeakyReLU = nn.LeakyReLU(.2)
    def forward(self, x):
        h = self.LeakyReLU(self.FC_hidden(x))
        h = self.LeakyReLU(self.FC_hidden2(h))
        x_hat = torch.sigmoid(self.FC_output(h)) # 흑백이므로 0(검정색), 1(흰색)
        return x_hat

In [ ]:
class Model(nn.Module):
    def __init__(self, Encoder, Decoder):
        super().__init__()
        self.Encoder = Encoder
        self.Decoder = Decoder
    def reparameterization(self, mean, var):
        epsilon = torch.randn_like(var).cuda() # Sampling Epsilon
        z = mean + var * epsilon            # reparameterization trick
        return z
    def forward(self, x):
        mean, log_var = self.Encoder(x)
        z = self.reparameterization(mean, torch.exp(0.5 * log_var)) # (log var -> var)
        x_hat = self.Decoder(z)
        return x_hat, mean, log_var

In [ ]:
# 각 모델을 연결
encoder = Encoder(input_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)
decoder = Decoder(output_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)
model = Model(Encoder=encoder, Decoder=decoder).cuda()


In [ ]:
BCE_loss = nn.BCELoss()
import torch.nn.functional as F
def loss_function(x, x_hat, mean, log_var):
    reproduction_error = F.binary_cross_entropy(x_hat, x, reduction="sum")
    KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reproduction_error + KLD
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
print("훈련을 시작합니다...")
model.train()
for epoch in range(epochs):
    overall_loss = 0
    for batch_idx, (x, _) in enumerate(train_loader):
        x = x.view(-1, x_dim) # 1차원 Vector화 (Flatten)
        x = x.cuda() # GPU용으로 변경
        optimizer.zero_grad() # 이전 루프 미분계산값 청소
        x_hat, mean, log_var = model(x)
        loss = loss_function(x, x_hat, mean, log_var)
        overall_loss += loss.item()
        loss.backward() # 현재 루프 미분값 계산
        optimizer.step() # 계산된 미분값 가중치에 업데이트
    print("Epoch", epoch + 1, "\tAverage Loss: ",
          overall_loss / (batch_idx*batch_size))
print("훈련이 종료됩니다...")

In [ ]:
import matplotlib.pyplot as plt
with torch.no_grad():
    for batch_x, (x, _) in enumerate(tqdm(test_loader)):
        x = x.view(-1, x_dim)
        x = x.cuda()
        x_hat, _, _ = model(x)
        break

In [ ]:
def show_image(x, idx):
    x = x.view(batch_size, 28, 28)
    fig = plt.figure
    plt.imshow(x[idx].cpu().numpy())
show_image(x, idx=0)

In [ ]:
show_image(x_hat, idx=0)